In [2]:
import os

from google.colab import userdata
token = userdata.get("KAGGLE_API_TOKEN")

!mkdir -p ~/.kaggle
!echo "{token}" > ~/.kaggle/access_token
!chmod 600 ~/.kaggle/access_token
!kaggle competitions list

ref                                                                           deadline             category         reward  teamCount  userHasEntered  
----------------------------------------------------------------------------  -------------------  --------  -------------  ---------  --------------  
https://www.kaggle.com/competitions/passenger-screening-algorithm-challenge   2017-12-15 23:59:00  Featured  1,500,000 Usd        518           False  
https://www.kaggle.com/competitions/zillow-prize-1                            2018-01-10 15:59:00  Featured  1,200,000 Usd       3770           False  
https://www.kaggle.com/competitions/data-science-bowl-2017                    2017-04-12 23:59:00  Featured  1,000,000 Usd       1972           False  
https://www.kaggle.com/competitions/vesuvius-challenge-ink-detection          2023-06-14 23:59:00  Featured  1,000,000 Usd       1249           False  
https://www.kaggle.com/competitions/arc-prize-2026-arc-agi-3                  2026-11-02

In [3]:
!pip install -q wandb
from google.colab import userdata
import os

os.environ["WANDB_API_KEY"] = userdata.get("WANDB_API_KEY")

import wandb
wandb.login()

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.
wandb: Currently logged in as: tasomamaladze123 (tasomamaladze123-none) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [4]:
from google.colab import userdata
TOKEN = userdata.get('GITHUB_TOKEN')
!git clone https://{TOKEN}@github.com/amama22/facial-expression-recognition
%cd facial-expression-recognition

Cloning into 'facial-expression-recognition'...
remote: Enumerating objects: 22, done.
remote: Counting objects: 100% (22/22), done.
remote: Compressing objects: 100% (17/17), done.
remote: Total 22 (delta 4), reused 13 (delta 2), pack-reused 0 (from 0)
Receiving objects: 100% (22/22), 6.97 KiB | 3.49 MiB/s, done.
Resolving deltas: 100% (4/4), done.
/content/facial-expression-recognition


In [5]:
!kaggle competitions download \
  -c challenges-in-representation-learning-facial-expression-recognition-challenge \
  -p data
!cd data && unzip -o -q "*.zip"
!ls -lh data

100% 285M/285M [00:14<00:00, 20.7MB/s]

total 952M
-rw-r--r-- 1 root root 286M Dec 11  2019 challenges-in-representation-learning-facial-expression-recognition-challenge.zip
-rw-r--r-- 1 root root 7.1K Dec 11  2019 example_submission.csv
-rw-r--r-- 1 root root  92M Dec 11  2019 fer2013.tar.gz
-rw-r--r-- 1 root root 288M Dec 11  2019 icml_face_data.csv
-rw-r--r-- 1 root root  58M Dec 11  2019 test.csv
-rw-r--r-- 1 root root 230M Dec 11  2019 train.csv


In [6]:
!cd data && tar -xzf fer2013.tar.gz

import glob
paths = glob.glob("data/**/fer2013.csv", recursive=True)
print(paths)

['data/fer2013/fer2013.csv']


In [7]:
import pandas as pd
df = pd.read_csv(paths[0])
print(df.shape)
print(df.columns.tolist())
print(df.head(3))
print(df["Usage"].value_counts())
print(df["emotion"].value_counts().sort_index())

(35887, 3)
['emotion', 'pixels', 'Usage']
   emotion                                             pixels     Usage
0        0  70 80 82 72 58 58 60 63 54 58 60 48 89 115 121...  Training
1        0  151 150 147 155 148 133 111 140 170 174 182 15...  Training
2        2  231 212 156 164 174 138 161 173 182 200 106 38...  Training
Usage
Training       28709
PublicTest      3589
PrivateTest     3589
Name: count, dtype: int64
emotion
0    4953
1     547
2    5121
3    8989
4    6077
5    4002
6    6198
Name: count, dtype: int64


In [8]:
%%writefile src/data.py
import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import transforms
from PIL import Image

EMOTIONS = {0: "Angry", 1: "Disgust", 2: "Fear", 3: "Happy",
            4: "Sad", 5: "Surprise", 6: "Neutral"}
NUM_CLASSES = 7
IMG_SIZE = 48

USAGE_TO_SPLIT = {"Training": "train", "PublicTest": "val", "PrivateTest": "test"}

FER_MEAN = (0.5077,)
FER_STD = (0.2550,)


def load_fer_arrays(csv_path):
    df = pd.read_csv(csv_path)
    images = np.stack(
        [np.asarray(p.split(), dtype=np.uint8) for p in df["pixels"]]
    ).reshape(-1, IMG_SIZE, IMG_SIZE)
    labels = df["emotion"].to_numpy(dtype=np.int64)
    usage = df["Usage"].to_numpy()
    return images, labels, usage


def split_arrays(images, labels, usage):
    splits = {}
    for usage_name, split_name in USAGE_TO_SPLIT.items():
        mask = usage == usage_name
        splits[split_name] = (images[mask], labels[mask])
    return splits


def compute_mean_std(images):
    x = images.astype(np.float32) / 255.0
    return float(x.mean()), float(x.std())


class FER2013Dataset(Dataset):
    def __init__(self, images, labels, transform=None):
        self.images = images
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img = Image.fromarray(self.images[idx], mode="L")
        if self.transform is not None:
            img = self.transform(img)
        return img, int(self.labels[idx])


def build_transforms(augment=False):
    tfms = []
    if augment:
        tfms += [
            transforms.RandomHorizontalFlip(),
            transforms.RandomRotation(10),
            transforms.RandomAffine(degrees=0, translate=(0.1, 0.1)),
        ]
    tfms += [transforms.ToTensor(), transforms.Normalize(FER_MEAN, FER_STD)]
    return transforms.Compose(tfms)


def class_weights(labels, num_classes=NUM_CLASSES):
    counts = np.bincount(labels, minlength=num_classes).astype(np.float32)
    weights = counts.sum() / (num_classes * counts)
    return torch.tensor(weights, dtype=torch.float32)


def make_weighted_sampler(labels, num_classes=NUM_CLASSES):
    counts = np.bincount(labels, minlength=num_classes).astype(np.float32)
    per_class_w = 1.0 / counts
    sample_w = per_class_w[labels]
    return WeightedRandomSampler(
        weights=torch.as_tensor(sample_w, dtype=torch.double),
        num_samples=len(sample_w),
        replacement=True,
    )


def get_dataloaders(csv_path, batch_size=64, augment=True, num_workers=2,
                    use_weighted_sampler=False):
    images, labels, usage = load_fer_arrays(csv_path)
    splits = split_arrays(images, labels, usage)

    train_ds = FER2013Dataset(*splits["train"], transform=build_transforms(augment))
    val_ds   = FER2013Dataset(*splits["val"], transform=build_transforms(False))
    test_ds  = FER2013Dataset(*splits["test"], transform=build_transforms(False))

    if use_weighted_sampler:
        sampler = make_weighted_sampler(splits["train"][1])
        train_loader = DataLoader(train_ds, batch_size=batch_size, sampler=sampler,
                                  num_workers=num_workers, pin_memory=True)
    else:
        train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True,
                                  num_workers=num_workers, pin_memory=True)

    val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False,
                            num_workers=num_workers, pin_memory=True)
    test_loader = DataLoader(test_ds, batch_size=batch_size, shuffle=False,
                             num_workers=num_workers, pin_memory=True)
    return train_loader, val_loader, test_loader

Overwriting src/data.py


In [9]:
import sys; sys.path.append("src")
import importlib, data; importlib.reload(data)

train_loader, val_loader, test_loader = data.get_dataloaders(paths[0], batch_size=64, augment=False)
xb, yb = next(iter(train_loader))
print("batch:", xb.shape, xb.dtype, "range:", round(xb.min().item(),2), "to", round(xb.max().item(),2))
print("labels:", yb[:8].tolist())
print("sizes:", len(train_loader.dataset), len(val_loader.dataset), len(test_loader.dataset))
print("mean/std:", data.compute_mean_std(data.load_fer_arrays(paths[0])[0]))
print("class weights:", data.class_weights(data.load_fer_arrays(paths[0])[1]).round(decimals=2).tolist())

batch: torch.Size([64, 1, 48, 48]) torch.float32 range: -1.99 to 1.93
labels: [4, 3, 6, 2, 5, 2, 3, 5]
sizes: 28709 3589 3589
mean/std: (0.507394552230835, 0.25512927770614624)
class weights: [1.0399999618530273, 9.369999885559082, 1.0, 0.5699999928474426, 0.8399999737739563, 1.2799999713897705, 0.8299999833106995]


In [10]:
#!git config --global user.email "amama22@freeuni.edu.ge"
#!git config --global user.name "amama22"
#!git add src/data.py
#!git commit -m "Add FER data pipeline"
#!git push

In [11]:
%%writefile src/models.py
import torch.nn as nn


class TinyCNN(nn.Module):
    def __init__(self, num_classes=7):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 16, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
        )
        self.classifier = nn.Linear(32 * 12 * 12, num_classes)

    def forward(self, x):
        x = self.features(x)
        x = x.flatten(1)
        return self.classifier(x)


def count_params(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

Overwriting src/models.py


In [12]:
%%writefile src/utils.py
import math
import numpy as np
import torch
import torch.nn as nn


def set_seed(seed=42):
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


@torch.no_grad()
def initial_loss_check(model, loader, device, num_classes=7):
    model.eval()
    xb, yb = next(iter(loader))
    xb, yb = xb.to(device), yb.to(device)
    loss = nn.functional.cross_entropy(model(xb), yb).item()
    expected = math.log(num_classes)
    print(f"initial loss = {loss:.3f}  (expected ~{expected:.3f})")
    return loss


def overfit_one_batch(model, loader, device, steps=200, lr=1e-3):
    model.train()
    xb, yb = next(iter(loader))
    xb, yb = xb.to(device), yb.to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    for i in range(steps):
        opt.zero_grad()
        out = model(xb)
        loss = nn.functional.cross_entropy(out, yb)
        loss.backward()
        opt.step()
        if (i + 1) % 50 == 0:
            acc = (out.argmax(1) == yb).float().mean().item()
            print(f"step {i + 1:4d}  loss {loss.item():.4f}  acc {acc:.3f}")
    return loss.item()

Overwriting src/utils.py


In [13]:
#!git add src/models.py
#!git add src/utils.py
#!git commit -m "Add tinyCNN + utils"
#!git push

In [14]:
import importlib, models, utils
importlib.reload(models); importlib.reload(utils)
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

utils.set_seed(42)
model = models.TinyCNN().to(device)
print("trainable params:", models.count_params(model))

utils.initial_loss_check(model, train_loader, device)
utils.overfit_one_batch(model, train_loader, device)

device: cuda
trainable params: 37063
initial loss = 1.945  (expected ~1.946)
step   50  loss 0.0023  acc 1.000
step  100  loss 0.0006  acc 1.000
step  150  loss 0.0004  acc 1.000
step  200  loss 0.0003  acc 1.000


0.00029352327692322433

In [15]:
%%writefile src/train.py

import numpy as np
import torch
import torch.nn as nn
from sklearn.metrics import f1_score
import wandb

import data as data_mod
import models as models_mod
import utils as utils_mod

ARCHITECTURES = {
    "TinyCNN": models_mod.TinyCNN,
}


def build_model(config, device):
    arch = ARCHITECTURES[config["arch"]]
    return arch(num_classes=data_mod.NUM_CLASSES).to(device)


def build_optimizer(model, config):
    wd = config.get("weight_decay", 0.0)
    if config["optimizer"] == "adam":
        return torch.optim.Adam(model.parameters(), lr=config["lr"], weight_decay=wd)
    if config["optimizer"] == "sgd":
        return torch.optim.SGD(model.parameters(), lr=config["lr"], momentum=0.9, weight_decay=wd)
    raise ValueError(f"unknown optimizer: {config['optimizer']}")


def train_one_epoch(model, loader, device, criterion, optimizer):
    model.train()
    total, correct, loss_sum = 0, 0, 0.0
    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        out = model(xb)
        loss = criterion(out, yb)
        loss.backward()
        optimizer.step()
        loss_sum += loss.item() * xb.size(0)
        correct += (out.argmax(1) == yb).sum().item()
        total += xb.size(0)
    return loss_sum / total, correct / total


@torch.no_grad()
def evaluate(model, loader, device, criterion):
    model.eval()
    total, correct, loss_sum = 0, 0, 0.0
    all_preds, all_targets = [], []
    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)
        out = model(xb)
        loss = criterion(out, yb)
        loss_sum += loss.item() * xb.size(0)
        preds = out.argmax(1)
        correct += (preds == yb).sum().item()
        total += xb.size(0)
        all_preds.append(preds.cpu().numpy())
        all_targets.append(yb.cpu().numpy())
    preds = np.concatenate(all_preds)
    targets = np.concatenate(all_targets)
    macro_f1 = f1_score(targets, preds, average="macro")
    return loss_sum / total, correct / total, macro_f1, preds, targets


def run_experiment(config, csv_path, project="fer2013", run_name=None):
    device = "cuda" if torch.cuda.is_available() else "cpu"
    utils_mod.set_seed(config.get("seed", 42))

    train_loader, val_loader, test_loader = data_mod.get_dataloaders(
        csv_path,
        batch_size=config["batch_size"],
        augment=config.get("augment", False),
        use_weighted_sampler=config.get("weighted_sampler", False),
    )

    model = build_model(config, device)
    optimizer = build_optimizer(model, config)

    if config.get("class_weighted_loss", False):
        _, labels, usage = data_mod.load_fer_arrays(csv_path)
        weight = data_mod.class_weights(labels[usage == "Training"]).to(device)
        criterion = nn.CrossEntropyLoss(weight=weight)
    else:
        criterion = nn.CrossEntropyLoss()

    run = wandb.init(
        project=project,
        name=run_name or f"{config['arch']}_lr{config['lr']}_bs{config['batch_size']}",
        group=config["arch"],
        config=config,
        reinit=True,
    )

    best_val_acc, best_state = 0.0, None
    for epoch in range(1, config["epochs"] + 1):
        tr_loss, tr_acc = train_one_epoch(model, train_loader, device, criterion, optimizer)
        val_loss, val_acc, val_f1, _, _ = evaluate(model, val_loader, device, criterion)

        wandb.log({
            "epoch": epoch,
            "train/loss": tr_loss, "train/acc": tr_acc,
            "val/loss": val_loss, "val/acc": val_acc, "val/macro_f1": val_f1,
            "lr": optimizer.param_groups[0]["lr"],
        })

        print(f"epoch {epoch:02d}  train_loss {tr_loss:.3f} acc {tr_acc:.3f} | "
              f"val_loss {val_loss:.3f} acc {val_acc:.3f} f1 {val_f1:.3f}")

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}

    if best_state is not None:
        model.load_state_dict(best_state)

    test_loss, test_acc, test_f1, test_preds, test_targets = evaluate(
        model, test_loader, device, criterion)

    wandb.summary["best_val_acc"] = best_val_acc
    wandb.summary["test_acc"] = test_acc
    wandb.summary["test_macro_f1"] = test_f1

    wandb.log({
        "confusion_matrix": wandb.plot.confusion_matrix(
            y_true=test_targets.tolist(),
            preds=test_preds.tolist(),
            class_names=[data_mod.EMOTIONS[i] for i in range(data_mod.NUM_CLASSES)],
        )
    })

    print(f"\nTEST  acc {test_acc:.3f}  macro_f1 {test_f1:.3f}")

    wandb.finish()

    return {
        "best_val_acc": best_val_acc,
        "test_acc": test_acc,
        "test_macro_f1": test_f1
    }

Overwriting src/train.py


In [16]:
import importlib, train
importlib.reload(train)

config = {
    "arch": "TinyCNN",
    "lr": 1e-3,
    "batch_size": 64,
    "epochs": 20,
    "optimizer": "adam",
    "weight_decay": 0.0,
    "augment": False,
    "class_weighted_loss": False,
    "weighted_sampler": False,
    "seed": 42,
}
train.run_experiment(config, paths[0], project="fer2013", run_name="A_tinycnn_baseline")

wandb: WARNING Using a boolean value for 'reinit' is deprecated. Use 'return_previous' or 'finish_previous' instead.


epoch 01  train_loss 1.582 acc 0.388 | val_loss 1.447 acc 0.446 f1 0.350
epoch 02  train_loss 1.389 acc 0.475 | val_loss 1.382 acc 0.477 f1 0.394
epoch 03  train_loss 1.286 acc 0.519 | val_loss 1.349 acc 0.482 f1 0.428
epoch 04  train_loss 1.212 acc 0.549 | val_loss 1.307 acc 0.507 f1 0.450
epoch 05  train_loss 1.150 acc 0.575 | val_loss 1.310 acc 0.511 f1 0.482
epoch 06  train_loss 1.095 acc 0.597 | val_loss 1.320 acc 0.515 f1 0.500
epoch 07  train_loss 1.052 acc 0.614 | val_loss 1.331 acc 0.514 f1 0.486
epoch 08  train_loss 1.011 acc 0.630 | val_loss 1.340 acc 0.510 f1 0.496
epoch 09  train_loss 0.974 acc 0.644 | val_loss 1.376 acc 0.508 f1 0.490
epoch 10  train_loss 0.945 acc 0.654 | val_loss 1.376 acc 0.508 f1 0.502
epoch 11  train_loss 0.917 acc 0.664 | val_loss 1.417 acc 0.504 f1 0.490
epoch 12  train_loss 0.891 acc 0.676 | val_loss 1.451 acc 0.508 f1 0.490
epoch 13  train_loss 0.866 acc 0.688 | val_loss 1.445 acc 0.505 f1 0.491
epoch 14  train_loss 0.848 acc 0.693 | val_loss 1.4

epoch,▁▁▂▂▂▃▃▄▄▄▅▅▅▆▆▇▇▇██
lr,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
train/acc,▁▃▄▄▅▅▆▆▆▆▇▇▇▇▇█████
train/loss,█▆▆▅▄▄▄▃▃▃▂▂▂▂▂▁▁▁▁▁
val/acc,▁▄▅▇████▇▇▇▇▇▇▆▆▇▆▆▆
val/loss,▄▃▂▁▁▁▂▂▂▂▃▄▄▅▅▆▆▇██
val/macro_f1,▁▃▅▆▇█▇█▇█▇▇██▇▇█▇▇▇
best_val_acc,0.51519
epoch,20
lr,0.001
test_acc,0.51379


{'best_val_acc': 0.5151852883811646,
 'test_acc': 0.513792142658122,
 'test_macro_f1': 0.485802539925494}

In [18]:
#!git config --global user.email "amama22@freeuni.edu.ge"
#!git config --global user.name "amama22"
#!git add src/train.py
#!git commit -m "Add training loop with macro-F1 and confusion matrix"
#!git push

[main d883ee5] Add training loop with macro-F1 and confusion matrix
 1 file changed, 142 insertions(+)
Enumerating objects: 7, done.
Counting objects: 100% (7/7), done.
Delta compression using up to 2 threads
Compressing objects: 100% (4/4), done.
Writing objects: 100% (4/4), 1.90 KiB | 1.90 MiB/s, done.
Total 4 (delta 2), reused 0 (delta 0), pack-reused 0
remote: Resolving deltas: 100% (2/2), completed with 2 local objects.
To https://github.com/amama22/facial-expression-recognition
   1f59ea5..d883ee5  main -> main


In [20]:
config = {
    "arch": "TinyCNN",
    "lr": 1e-3, "batch_size": 64, "epochs": 20, "optimizer": "adam",
    "weight_decay": 1e-4,        # added: L2 regularization
    "augment": True,             # added: data augmentation
    "class_weighted_loss": False, "weighted_sampler": False,
    "model_kwargs": {},
    "seed": 42,
}
train.run_experiment(config, paths[0], project="fer2013", run_name="A_tinycnn_regularized")

epoch 01  train_loss 1.709 acc 0.313 | val_loss 1.540 acc 0.415 f1 0.314
epoch 02  train_loss 1.594 acc 0.380 | val_loss 1.473 acc 0.441 f1 0.332
epoch 03  train_loss 1.516 acc 0.417 | val_loss 1.409 acc 0.456 f1 0.358
epoch 04  train_loss 1.470 acc 0.437 | val_loss 1.344 acc 0.487 f1 0.389
epoch 05  train_loss 1.433 acc 0.453 | val_loss 1.327 acc 0.496 f1 0.410
epoch 06  train_loss 1.401 acc 0.468 | val_loss 1.316 acc 0.505 f1 0.438
epoch 07  train_loss 1.386 acc 0.475 | val_loss 1.294 acc 0.507 f1 0.429
epoch 08  train_loss 1.368 acc 0.485 | val_loss 1.297 acc 0.500 f1 0.426
epoch 09  train_loss 1.362 acc 0.487 | val_loss 1.285 acc 0.508 f1 0.445
epoch 10  train_loss 1.349 acc 0.488 | val_loss 1.274 acc 0.516 f1 0.451
epoch 11  train_loss 1.338 acc 0.495 | val_loss 1.264 acc 0.510 f1 0.436
epoch 12  train_loss 1.335 acc 0.493 | val_loss 1.274 acc 0.517 f1 0.456
epoch 13  train_loss 1.324 acc 0.497 | val_loss 1.268 acc 0.515 f1 0.448
epoch 14  train_loss 1.319 acc 0.499 | val_loss 1.2

epoch,▁▁▂▂▂▃▃▄▄▄▅▅▅▆▆▇▇▇██
lr,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
train/acc,▁▃▅▅▆▆▇▇▇▇▇▇▇███████
train/loss,█▆▅▄▃▃▃▂▂▂▂▂▂▁▁▁▁▁▁▁
val/acc,▁▃▃▅▆▆▆▆▆▇▇▇▇█▇▇▇███
val/loss,█▆▅▃▃▃▂▂▂▂▂▂▂▁▁▁▁▁▁▁
val/macro_f1,▁▂▃▄▅▆▆▆▆▇▆▇▇█▇▇▇███
best_val_acc,0.53385
epoch,20
lr,0.001
test_acc,0.52856


{'best_val_acc': 0.5338534410699359,
 'test_acc': 0.5285594873223739,
 'test_macro_f1': 0.454434953975376}

In [22]:
!git fetch origin && git reset --hard origin/main
import importlib, models, train
importlib.reload(models); importlib.reload(train)
print("DeeperCNN registered:", "DeeperCNN" in train.ARCHITECTURES)
print("DeeperCNN in models:", hasattr(models, "DeeperCNN"))

remote: Enumerating objects: 12, done.
remote: Counting objects: 100% (12/12), done.
remote: Compressing objects: 100% (8/8), done.
remote: Total 8 (delta 4), reused 0 (delta 0), pack-reused 0 (from 0)
Unpacking objects: 100% (8/8), 2.54 KiB | 1.27 MiB/s, done.
From https://github.com/amama22/facial-expression-recognition
   d883ee5..a690750  main       -> origin/main
HEAD is now at a690750 Update build_model to accept additional model kwargs
DeeperCNN registered: True
DeeperCNN in models: True


In [23]:
import importlib, train, models
importlib.reload(models); importlib.reload(train)

config = {
    "arch": "DeeperCNN",
    "lr": 1e-3, "batch_size": 64, "epochs": 20, "optimizer": "adam",
    "weight_decay": 0.0,
    "augment": False,
    "class_weighted_loss": False, "weighted_sampler": False,
    "model_kwargs": {"p_drop": 0.3},
    "seed": 42,
}
train.run_experiment(config, paths[0], project="fer2013", run_name="B_deepercnn_baseline")

epoch 01  train_loss 1.513 acc 0.410 | val_loss 1.303 acc 0.493 f1 0.412
epoch 02  train_loss 1.255 acc 0.520 | val_loss 1.173 acc 0.544 f1 0.464
epoch 03  train_loss 1.164 acc 0.558 | val_loss 1.122 acc 0.566 f1 0.512
epoch 04  train_loss 1.100 acc 0.580 | val_loss 1.096 acc 0.581 f1 0.535
epoch 05  train_loss 1.045 acc 0.605 | val_loss 1.041 acc 0.608 f1 0.576
epoch 06  train_loss 1.004 acc 0.620 | val_loss 1.062 acc 0.597 f1 0.556
epoch 07  train_loss 0.968 acc 0.635 | val_loss 1.017 acc 0.614 f1 0.583
epoch 08  train_loss 0.928 acc 0.649 | val_loss 1.004 acc 0.631 f1 0.610
epoch 09  train_loss 0.891 acc 0.666 | val_loss 1.003 acc 0.627 f1 0.599
epoch 10  train_loss 0.857 acc 0.677 | val_loss 0.982 acc 0.629 f1 0.605
epoch 11  train_loss 0.810 acc 0.698 | val_loss 1.001 acc 0.641 f1 0.619
epoch 12  train_loss 0.788 acc 0.708 | val_loss 0.984 acc 0.646 f1 0.628
epoch 13  train_loss 0.744 acc 0.721 | val_loss 0.991 acc 0.644 f1 0.628
epoch 14  train_loss 0.714 acc 0.735 | val_loss 1.0

epoch,▁▁▂▂▂▃▃▄▄▄▅▅▅▆▆▇▇▇██
lr,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
train/acc,▁▃▄▄▄▅▅▅▆▆▆▆▇▇▇▇▇███
train/loss,█▆▅▅▅▄▄▄▄▃▃▃▂▂▂▂▂▁▁▁
val/acc,▁▃▄▅▆▅▆▇▇▇▇█▇▇██████
val/loss,█▅▄▃▂▃▂▁▁▁▁▁▁▁▂▂▂▂▃▃
val/macro_f1,▁▃▄▅▆▅▆▇▇▇▇▇▇▇▇███▇█
best_val_acc,0.65561
epoch,20
lr,0.001
test_acc,0.6701


{'best_val_acc': 0.6556143772638618,
 'test_acc': 0.6701030927835051,
 'test_macro_f1': 0.6615915889143962}

In [26]:
!git fetch origin && git reset --hard origin/main
import importlib, models, train, data
importlib.reload(models); importlib.reload(train)

config = {
    "arch": "SmallResNet",
    "lr": 1e-3, "batch_size": 64, "epochs": 20, "optimizer": "adam",
    "weight_decay": 0.0, "augment": False,
    "class_weighted_loss": False, "weighted_sampler": False,
    "model_kwargs": {"p_drop": 0.3},
    "seed": 42,
}
train.run_experiment(config, paths[0], project="fer2013", run_name="C_resnet_baseline")

remote: Enumerating objects: 7, done.
remote: Counting objects: 100% (7/7), done.
remote: Compressing objects: 100% (4/4), done.
remote: Total 4 (delta 2), reused 0 (delta 0), pack-reused 0 (from 0)
Unpacking objects: 100% (4/4), 1.16 KiB | 1.16 MiB/s, done.
From https://github.com/amama22/facial-expression-recognition
   262a59a..b5da9d6  main       -> origin/main
HEAD is now at b5da9d6 Refactor DataLoader initialization


epoch,▁
lr,▁
train/acc,▁
train/loss,▁
val/acc,▁
val/loss,▁
val/macro_f1,▁
epoch,1
lr,0.001
train/acc,0.27824
train/loss,1.76774


epoch 01  train_loss 1.778 acc 0.271 | val_loss 1.693 acc 0.325 f1 0.242
epoch 02  train_loss 1.471 acc 0.425 | val_loss 1.353 acc 0.478 f1 0.376
epoch 03  train_loss 1.276 acc 0.509 | val_loss 1.248 acc 0.527 f1 0.422
epoch 04  train_loss 1.172 acc 0.551 | val_loss 1.228 acc 0.539 f1 0.459
epoch 05  train_loss 1.097 acc 0.587 | val_loss 1.305 acc 0.519 f1 0.426
epoch 06  train_loss 1.035 acc 0.609 | val_loss 1.088 acc 0.578 f1 0.527
epoch 07  train_loss 0.975 acc 0.632 | val_loss 1.092 acc 0.581 f1 0.517
epoch 08  train_loss 0.915 acc 0.659 | val_loss 1.231 acc 0.558 f1 0.476
epoch 09  train_loss 0.850 acc 0.681 | val_loss 1.105 acc 0.600 f1 0.535
epoch 10  train_loss 0.763 acc 0.719 | val_loss 1.135 acc 0.597 f1 0.556
epoch 11  train_loss 0.670 acc 0.753 | val_loss 1.240 acc 0.590 f1 0.531
epoch 12  train_loss 0.555 acc 0.798 | val_loss 1.303 acc 0.598 f1 0.562
epoch 13  train_loss 0.440 acc 0.843 | val_loss 1.589 acc 0.559 f1 0.537
epoch 14  train_loss 0.321 acc 0.887 | val_loss 1.5

epoch,▁▁▂▂▂▃▃▄▄▄▅▅▅▆▆▇▇▇██
lr,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
train/acc,▁▃▃▄▄▄▅▅▅▅▆▆▇▇▇█████
train/loss,█▇▆▅▅▅▅▄▄▄▃▃▂▂▂▁▁▁▁▁
val/acc,▁▅▆▆▆▇█▇████▇████▇█▇
val/loss,▄▂▂▂▂▁▁▂▁▁▂▂▄▄▅▆▆▇▇█
val/macro_f1,▁▄▅▆▅▇▇▆▇█▇█▇▇███▇██
best_val_acc,0.60045
epoch,20
lr,0.001
test_acc,0.59766


{'best_val_acc': 0.6004458066313736,
 'test_acc': 0.5976595151852884,
 'test_macro_f1': 0.5139716181952916}

In [27]:
config = {
    "arch": "SmallResNet",
    "lr": 1e-3, "batch_size": 64, "epochs": 30, "optimizer": "adam",
    "weight_decay": 1e-4,
    "augment": True,                 # the big lever against its overfitting
    "class_weighted_loss": False, "weighted_sampler": False,
    "p_drop": 0.3,
    "seed": 42,
}
train.run_experiment(config, paths[0], project="fer2013", run_name="C_resnet_regularized")

epoch 01  train_loss 1.769 acc 0.272 | val_loss 1.618 acc 0.354 f1 0.227
epoch 02  train_loss 1.482 acc 0.420 | val_loss 1.447 acc 0.411 f1 0.326
epoch 03  train_loss 1.340 acc 0.486 | val_loss 1.385 acc 0.477 f1 0.386
epoch 04  train_loss 1.268 acc 0.518 | val_loss 1.299 acc 0.513 f1 0.380
epoch 05  train_loss 1.214 acc 0.537 | val_loss 1.376 acc 0.481 f1 0.377
epoch 06  train_loss 1.177 acc 0.552 | val_loss 1.320 acc 0.507 f1 0.444
epoch 07  train_loss 1.136 acc 0.568 | val_loss 1.160 acc 0.562 f1 0.454
epoch 08  train_loss 1.111 acc 0.580 | val_loss 1.327 acc 0.528 f1 0.410
epoch 09  train_loss 1.095 acc 0.587 | val_loss 1.201 acc 0.555 f1 0.451
epoch 10  train_loss 1.067 acc 0.597 | val_loss 1.144 acc 0.568 f1 0.512
epoch 11  train_loss 1.048 acc 0.606 | val_loss 1.154 acc 0.568 f1 0.459
epoch 12  train_loss 1.031 acc 0.611 | val_loss 1.049 acc 0.609 f1 0.546
epoch 13  train_loss 1.019 acc 0.616 | val_loss 1.041 acc 0.609 f1 0.534
epoch 14  train_loss 1.007 acc 0.621 | val_loss 1.0

epoch,▁▁▁▂▂▂▂▃▃▃▃▄▄▄▄▅▅▅▅▆▆▆▆▇▇▇▇███
lr,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
train/acc,▁▄▅▅▆▆▆▆▆▇▇▇▇▇▇▇▇▇▇▇██████████
train/loss,█▆▅▄▄▃▃▃▃▃▃▂▂▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁
val/acc,▁▂▄▅▄▅▆▅▆▆▆▇▇▇▇▇█▇▇▇█▇█▇███▇██
val/loss,█▆▅▄▅▅▃▅▃▃▃▂▂▂▂▂▁▂▂▂▂▂▂▂▁▁▁▂▁▁
val/macro_f1,▁▃▄▄▄▅▅▄▅▆▅▇▇▆▇▇█▇▇▇▇▇█▇▇█████
best_val_acc,0.64503
epoch,30
lr,0.001
test_acc,0.65617


{'best_val_acc': 0.6450264697687378,
 'test_acc': 0.6561716355530789,
 'test_macro_f1': 0.6213081329991424}

In [ ]:
!git fetch origin && git reset --hard origin/main
import importlib, models, train, data

In [32]:
!git fetch origin && git reset --hard origin/main
import importlib, models, train, data
import wandb, functools

sweep_config = {
    "method": "bayes",
    "metric": {"name": "val/acc", "goal": "maximize"},
    "parameters": {
        "arch":         {"value": "SmallResNet"},
        "epochs":       {"value": 15},
        "optimizer":    {"values": ["adam", "sgd"]},
        "lr":           {"distribution": "log_uniform_values", "min": 2e-4, "max": 3e-3},
        "batch_size":   {"values": [64, 128]},
        "weight_decay": {"values": [0.0, 1e-4, 5e-4]},
        "augment":      {"value": True},
        "p_drop":       {"values": [0.2, 0.3, 0.4]},
        "scheduler":    {"values": ["cosine", "none"]},
    },
}
defaults = {"arch": "SmallResNet", "epochs": 15, "optimizer": "adam", "lr": 1e-3,
            "batch_size": 64, "weight_decay": 1e-4, "augment": True, "p_drop": 0.3,
            "scheduler": "cosine", "seed": 42,
            "class_weighted_loss": False, "weighted_sampler": False}

sweep_id = wandb.sweep(sweep_config, project="fer2013")
wandb.agent(sweep_id, function=functools.partial(train.sweep_train, paths[0], defaults), count=8)

HEAD is now at ee3b7e4 Refactor training functions and integrate wandb
Create sweep with ID: cw4k2wfe
Sweep URL: https://wandb.ai/tasomamaladze123-none/fer2013/sweeps/cw4k2wfe


wandb: Agent Starting Run: lcwkkg4e with config:
wandb: 	arch: SmallResNet
wandb: 	augment: True
wandb: 	batch_size: 128
wandb: 	epochs: 15
wandb: 	lr: 0.002202211037778878
wandb: 	optimizer: adam
wandb: 	p_drop: 0.3
wandb: 	scheduler: none
wandb: 	weight_decay: 0
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.


epoch 01  train_loss 1.768 acc 0.273 | val_loss 1.673 acc 0.318 f1 0.210
epoch 02  train_loss 1.541 acc 0.393 | val_loss 1.408 acc 0.446 f1 0.340
epoch 03  train_loss 1.371 acc 0.469 | val_loss 1.465 acc 0.429 f1 0.357
epoch 04  train_loss 1.268 acc 0.514 | val_loss 1.276 acc 0.508 f1 0.390
epoch 05  train_loss 1.197 acc 0.548 | val_loss 1.183 acc 0.550 f1 0.445
epoch 06  train_loss 1.135 acc 0.569 | val_loss 1.167 acc 0.560 f1 0.479
epoch 07  train_loss 1.097 acc 0.584 | val_loss 1.122 acc 0.576 f1 0.477
epoch 08  train_loss 1.060 acc 0.600 | val_loss 1.217 acc 0.556 f1 0.446
epoch 09  train_loss 1.038 acc 0.606 | val_loss 1.115 acc 0.587 f1 0.501
epoch 10  train_loss 1.011 acc 0.621 | val_loss 1.081 acc 0.606 f1 0.534
epoch 11  train_loss 0.989 acc 0.628 | val_loss 1.057 acc 0.594 f1 0.526
epoch 12  train_loss 0.971 acc 0.631 | val_loss 1.034 acc 0.619 f1 0.563
epoch 13  train_loss 0.949 acc 0.640 | val_loss 1.036 acc 0.614 f1 0.552
epoch 14  train_loss 0.936 acc 0.647 | val_loss 1.0

epoch,▁▁▂▃▃▃▄▅▅▅▆▇▇▇█
lr,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
train/acc,▁▃▅▅▆▆▇▇▇▇▇████
train/loss,█▆▅▄▃▃▂▂▂▂▂▁▁▁▁
val/acc,▁▄▄▅▆▇▇▆▇█▇████
val/loss,█▅▆▄▃▃▂▃▂▂▂▁▁▁▁
val/macro_f1,▁▃▄▄▅▆▆▅▆▇▇█▇██
best_val_acc,0.62051
epoch,15
lr,0.0022
test_acc,0.63862


wandb: Sweep Agent: Waiting for job.
wandb: Job received.
wandb: Agent Starting Run: 8qtzstiy with config:
wandb: 	arch: SmallResNet
wandb: 	augment: True
wandb: 	batch_size: 128
wandb: 	epochs: 15
wandb: 	lr: 0.0006569736567587754
wandb: 	optimizer: adam
wandb: 	p_drop: 0.3
wandb: 	scheduler: none
wandb: 	weight_decay: 0.0005
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.


...

epoch 01  train_loss 1.784 acc 0.264 | val_loss 1.755 acc 0.239 f1 0.144
epoch 02  train_loss 1.569 acc 0.375 | val_loss 1.731 acc 0.334 f1 0.248
epoch 03  train_loss 1.449 acc 0.433 | val_loss 1.445 acc 0.447 f1 0.332
epoch 04  train_loss 1.393 acc 0.465 | val_loss 1.442 acc 0.452 f1 0.334
epoch 05  train_loss 1.343 acc 0.484 | val_loss 1.346 acc 0.482 f1 0.383
epoch 06  train_loss 1.307 acc 0.500 | val_loss 1.448 acc 0.444 f1 0.318
epoch 07  train_loss 1.273 acc 0.515 | val_loss 1.271 acc 0.498 f1 0.396
epoch 08  train_loss 1.239 acc 0.525 | val_loss 1.416 acc 0.490 f1 0.355
epoch 09  train_loss 1.215 acc 0.537 | val_loss 1.358 acc 0.494 f1 0.350
epoch 10  train_loss 1.188 acc 0.548 | val_loss 1.196 acc 0.539 f1 0.436
epoch 11  train_loss 1.173 acc 0.554 | val_loss 1.205 acc 0.539 f1 0.419
epoch 12  train_loss 1.158 acc 0.562 | val_loss 1.233 acc 0.543 f1 0.437
epoch 13  train_loss 1.147 acc 0.569 | val_loss 1.162 acc 0.560 f1 0.482
epoch 14  train_loss 1.132 acc 0.572 | val_loss 1.1

epoch,▁▁▂▃▃▃▄▅▅▅▆▇▇▇█
lr,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
train/acc,▁▃▅▅▆▆▇▇▇▇▇████
train/loss,█▆▄▄▃▃▃▂▂▂▂▁▁▁▁
val/acc,▁▃▅▆▆▅▇▆▆▇▇████
val/loss,██▄▄▃▅▃▄▃▂▂▂▁▁▂
val/macro_f1,▁▃▅▅▆▅▆▅▅▇▇▇███
best_val_acc,0.56534
epoch,15
lr,0.00182
test_acc,0.57481


wandb: Sweep Agent: Waiting for job.
wandb: Job received.
wandb: Agent Starting Run: ojzsp35w with config:
wandb: 	arch: SmallResNet
wandb: 	augment: True
wandb: 	batch_size: 128
wandb: 	epochs: 15
wandb: 	lr: 0.001512888789380477
wandb: 	optimizer: adam
wandb: 	p_drop: 0.3
wandb: 	scheduler: none
wandb: 	weight_decay: 0.0005
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.


epoch 01  train_loss 1.755 acc 0.280 | val_loss 1.717 acc 0.276 f1 0.177
epoch 02  train_loss 1.538 acc 0.397 | val_loss 1.564 acc 0.391 f1 0.324
epoch 03  train_loss 1.399 acc 0.461 | val_loss 1.841 acc 0.365 f1 0.251
epoch 04  train_loss 1.330 acc 0.491 | val_loss 1.383 acc 0.463 f1 0.325
epoch 05  train_loss 1.284 acc 0.511 | val_loss 1.293 acc 0.509 f1 0.387
epoch 06  train_loss 1.233 acc 0.531 | val_loss 1.390 acc 0.462 f1 0.390
epoch 07  train_loss 1.197 acc 0.546 | val_loss 1.352 acc 0.476 f1 0.363
epoch 08  train_loss 1.167 acc 0.560 | val_loss 1.268 acc 0.518 f1 0.410
epoch 09  train_loss 1.146 acc 0.566 | val_loss 1.131 acc 0.567 f1 0.463
epoch 10  train_loss 1.124 acc 0.575 | val_loss 1.124 acc 0.572 f1 0.490
epoch 11  train_loss 1.107 acc 0.580 | val_loss 1.137 acc 0.569 f1 0.455
epoch 12  train_loss 1.087 acc 0.588 | val_loss 1.254 acc 0.535 f1 0.419
epoch 13  train_loss 1.076 acc 0.596 | val_loss 1.239 acc 0.542 f1 0.480
epoch 14  train_loss 1.067 acc 0.599 | val_loss 1.1

epoch,▁▁▂▃▃▃▄▅▅▅▆▇▇▇█
lr,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
train/acc,▁▄▅▆▆▆▇▇▇▇▇████
train/loss,█▆▄▄▃▃▂▂▂▂▂▁▁▁▁
val/acc,▁▄▃▅▇▅▆▇███▇▇██
val/loss,▇▅█▄▃▄▃▂▁▁▁▂▂▁▁
val/macro_f1,▁▄▃▄▅▆▅▆▇█▇▆▇██
best_val_acc,0.5723
epoch,15
lr,0.00151
test_acc,0.57676


In [33]:
best_config = {
    "arch": "SmallResNet",
    "lr": 0.0012, "batch_size": 64, "epochs": 45, "optimizer": "adam",
    "weight_decay": 5e-4, "augment": True, "p_drop": 0.2, "scheduler": "cosine",
    "class_weighted_loss": False, "weighted_sampler": False, "seed": 42,
}
train.run_experiment(best_config, paths[0], project="fer2013", run_name="C_resnet_final")

epoch 01  train_loss 1.760 acc 0.276 | val_loss 1.769 acc 0.323 f1 0.189
epoch 02  train_loss 1.486 acc 0.421 | val_loss 1.437 acc 0.449 f1 0.337
epoch 03  train_loss 1.385 acc 0.468 | val_loss 1.461 acc 0.453 f1 0.377
epoch 04  train_loss 1.328 acc 0.492 | val_loss 1.594 acc 0.433 f1 0.301
epoch 05  train_loss 1.280 acc 0.514 | val_loss 1.450 acc 0.438 f1 0.323
epoch 06  train_loss 1.240 acc 0.526 | val_loss 1.388 acc 0.468 f1 0.373
epoch 07  train_loss 1.200 acc 0.544 | val_loss 1.332 acc 0.485 f1 0.404
epoch 08  train_loss 1.167 acc 0.559 | val_loss 1.292 acc 0.525 f1 0.414
epoch 09  train_loss 1.143 acc 0.564 | val_loss 1.265 acc 0.532 f1 0.432
epoch 10  train_loss 1.115 acc 0.580 | val_loss 1.233 acc 0.534 f1 0.435
epoch 11  train_loss 1.097 acc 0.585 | val_loss 1.182 acc 0.544 f1 0.433
epoch 12  train_loss 1.079 acc 0.590 | val_loss 1.080 acc 0.591 f1 0.504
epoch 13  train_loss 1.062 acc 0.601 | val_loss 1.097 acc 0.589 f1 0.514
epoch 14  train_loss 1.049 acc 0.604 | val_loss 1.1

epoch,▁▁▁▁▂▂▂▂▂▃▃▃▃▃▃▄▄▄▄▄▅▅▅▅▅▅▆▆▆▆▆▆▇▇▇▇▇███
lr,████████▇▇▇▇▇▆▆▆▆▅▅▅▅▄▄▄▃▃▃▃▃▂▂▂▂▂▁▁▁▁▁▁
train/acc,▁▃▄▄▅▅▅▅▆▆▆▆▆▆▆▆▆▆▆▇▇▇▇▇▇▇▇▇▇▇▇█████████
train/loss,█▆▆▅▅▅▄▄▄▄▄▃▃▃▃▃▃▃▃▃▃▃▂▂▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁
val/acc,▁▄▄▃▃▄▄▅▅▅▆▆▆▇▆▇▇▇▇▇▇▇▇▇▇▇▇██▇██████████
val/loss,█▅▅▇▅▅▄▄▃▃▂▂▃▂▂▂▂▂▂▂▁▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
val/macro_f1,▁▃▄▃▃▄▄▅▅▅▆▆▆▇▇▆▆▇▇▇▇▇▇▇█▇▇██▇██████████
best_val_acc,0.67261
epoch,45
lr,0.0
test_acc,0.67735


{'best_val_acc': 0.6726107550849819,
 'test_acc': 0.6773474505433268,
 'test_macro_f1': 0.6473640519093854}

In [34]:
import wandb; wandb.finish()